In [0]:
# install dependencies
# %pip install -e ..
# %pip install git+https://github.com/end-to-end-mlops-databricks-3/marvelous@0.1.0

In [0]:
%pip install hotel_reservation-0.0.1-py3-none-any.whl

Processing ./hotel_reservation-0.0.1-py3-none-any.whl
hotel-reservation is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# restart python
%restart_python

In [0]:
# system path update, must be after %restart_python
# caution! This is not a great approach
# from pathlib import Path
# import sys
# sys.path.append(str(Path.cwd().parent / 'src'))

In [0]:
import os

import boto3
import mlflow
import numpy as np
import pandas as pd
from databricks import feature_engineering
from databricks.feature_engineering import FeatureFunction, FeatureLookup
from dotenv import load_dotenv
from lightgbm import LGBMClassifier
from mlflow.models import infer_signature
from pyspark.sql import SparkSession
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

from hotel_reservation.config import ProjectConfig
from hotel_reservation.utils import is_databricks

In [0]:
if not is_databricks():
    load_dotenv()
    profile = os.environ["PROFILE"]
    mlflow.set_tracking_uri(f"databricks://{profile}")
    mlflow.set_registry_uri(f"databricks-uc://{profile}")


config = ProjectConfig.from_yaml(config_path="../project_config.yml", env="dev")

In [0]:
spark = SparkSession.builder.getOrCreate()
fe = feature_engineering.FeatureEngineeringClient()

train_set = spark.table(f"{config.catalog_name}.{config.schema_name}.train_set")
test_set = spark.table(f"{config.catalog_name}.{config.schema_name}.test_set")

In [0]:
# create feature table with information about hotel reservations

feature_table_name = f"{config.catalog_name}.{config.schema_name}.hotel_reservation_features_demo"
lookup_features = ["no_of_previous_cancellations", "no_of_previous_bookings_not_canceled"]

In [0]:
# Option 1: feature engineering client
feature_table = fe.create_table(
    name=feature_table_name,
    primary_keys=["Booking_ID"],
    df=train_set[["Booking_ID"] + lookup_features],
    description="Hotel Reservation features table",
)

spark.sql(f"ALTER TABLE {feature_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

fe.write_table(
    name=feature_table_name,
    df=test_set[["Booking_ID"] + lookup_features],
    mode="merge",
)

2025/09/24 07:11:22 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['Booking_ID'] of table 'mlops_dev.nikhilko.hotel_reservation_features_demo' to NOT NULL.
2025/09/24 07:11:24 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['Booking_ID'] on table 'mlops_dev.nikhilko.hotel_reservation_features_demo'.
2025/09/24 07:11:30 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'mlops_dev.nikhilko.hotel_reservation_features_demo'.


In [0]:
# # create feature table with information about hotel reservations
# # Option 2: SQL

# spark.sql(f"""
#           CREATE OR REPLACE TABLE {feature_table_name}
#           (Booking_ID STRING NOT NULL, no_of_previous_cancellations INT, no_of_previous_bookings_not_canceled INT);
#           """)
# # primary key on Databricks is not enforced!
# try:
#     spark.sql(f"ALTER TABLE {feature_table_name} ADD CONSTRAINT hotel_reservation_pk_demo PRIMARY KEY(Booking_ID);")
# except AnalysisException:
#     pass
# spark.sql(f"ALTER TABLE {feature_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true);")
# spark.sql(f"""
#           INSERT INTO {feature_table_name}
#           SELECT Booking_ID, no_of_previous_cancellations, no_of_previous_bookings_not_canceled
#           FROM {config.catalog_name}.{config.schema_name}.train_set
#           """)
# spark.sql(f"""
#           INSERT INTO {feature_table_name}
#           SELECT Booking_ID, no_of_previous_cancellations, no_of_previous_bookings_not_canceled
#           FROM {config.catalog_name}.{config.schema_name}.test_set
#           """)

In [0]:
# create feature function
# docs: https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function

# problems with feature functions:
# functions are not versioned
# functions may behave differently depending on the runtime (and version of packages and python)
# there is no way to enforce python version & package versions for the function
# this is only supported from runtime 17
# advised to use only for simple calculations

function_name = f"{config.catalog_name}.{config.schema_name}.calculate_booking_value_demo"

In [0]:
# Option 1: with Python
spark.sql(f"""
        CREATE OR REPLACE FUNCTION {function_name}(
                avg_price_per_room DOUBLE,
                no_of_weekend_nights BIGINT,
                no_of_week_nights BIGINT
        )
        RETURNS DOUBLE
        LANGUAGE PYTHON AS
        $$
        if avg_price_per_room is None or no_of_weekend_nights is None or no_of_week_nights is None:
                return None
        return avg_price_per_room * (no_of_weekend_nights + no_of_week_nights)
        $$
""")

DataFrame[]

In [0]:
# # it is possible to define simple functions in sql only without python
# # Option 2
# spark.sql(f"""
#         CREATE OR REPLACE FUNCTION {function_name}_sql(
#                 avg_price_per_room DOUBLE,
#                 no_of_weekend_nights INT,
#                 no_of_week_nights INT
#         )
#         RETURNS DOUBLE
#         RETURN
#         CASE
#                 WHEN avg_price_per_room IS NULL
#                         OR no_of_weekend_nights IS NULL
#                         OR no_of_week_nights IS NULL
#                 THEN NULL
#                 ELSE avg_price_per_room * (no_of_weekend_nights + no_of_week_nights)
#         END

#         """)

In [0]:
# # Run the query and get the result as a DataFrame
# result_df = spark.sql(f"SELECT {function_name}_sql(90.1, 2, 2) AS booking_value")

# # Show the result in tabular form
# result_df.show()

In [0]:
# create a training set
training_set = fe.create_training_set(
    df=train_set.drop("no_of_previous_cancellations", "no_of_previous_bookings_not_canceled"),
    label=config.target,
    feature_lookups=[
        FeatureLookup(
            table_name=feature_table_name,
            feature_names=["no_of_previous_cancellations", "no_of_previous_bookings_not_canceled"],
            lookup_key="Booking_ID",
        ),
        FeatureFunction(
            udf_name=function_name,
            output_name="booking_value",
            input_bindings={
                "avg_price_per_room": "avg_price_per_room",
                "no_of_weekend_nights": "no_of_weekend_nights",
                "no_of_week_nights": "no_of_week_nights",
            },
        ),
    ],
    exclude_columns=["update_timestamp_utc"],
)

In [0]:
# Train & register a model
training_df = training_set.load_df().toPandas()
X_train = training_df[config.num_features + config.cat_features + ["booking_value"]]
y_train = training_df[config.target]

In [0]:
# Encode the label
labelEncoder = LabelEncoder()
y_train_encoded = labelEncoder.fit_transform(y_train)

In [0]:
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), config.cat_features)],
                remainder="passthrough",
            ),
        ),
        ("classifier", LGBMClassifier(**config.parameters)),
    ]
)

pipeline.fit(X_train, y_train_encoded)

[LightGBM] [Info] Number of positive: 19551, number of negative: 9469
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005291 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 915
[LightGBM] [Info] Number of data points in the train set: 29020, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.673708 -> initscore=0.725003
[LightGBM] [Info] Start training from score 0.725003
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [0]:
mlflow.set_experiment("/Shared/demo-model-fe")
with mlflow.start_run(
    run_name="demo-run-model-fe",
    tags={"git_sha": "1234567890abcd", "branch": "week2"},
    description="demo run for FE model logging",
) as run:
    # Log parameters and metrics
    run_id = run.info.run_id
    mlflow.log_param("model_type", "LightGBM with preprocessing")
    mlflow.log_params(config.parameters)

    # Log the model
    signature = infer_signature(model_input=X_train, model_output=pipeline.predict(X_train))
    fe.log_model(
        model=pipeline,
        flavor=mlflow.sklearn,
        artifact_path="lightgbm-pipeline-model-fe",
        training_set=training_set,
        signature=signature,
    )

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.h

In [0]:
model_name = f"{config.catalog_name}.{config.schema_name}.model_fe_demo"
model_version = mlflow.register_model(
    model_uri=f"runs:/{run_id}/lightgbm-pipeline-model-fe", name=model_name, tags={"git_sha": "1234567890abcd"}
)

Registered model 'mlops_dev.nikhilko.model_fe_demo' already exists. Creating a new version of this model...
2025/09/24 07:25:45 WARNING mlflow.tracking._model_registry.fluent: Run with id 6fe3235354d64ead932e73525ad03ece has no artifacts at artifact path 'lightgbm-pipeline-model-fe', registering model based on models:/m-04447cd799674f3dbad0cb00d06706fa instead


Uploading artifacts:   0%|          | 0/15 [00:00<?, ?it/s]

🔗 Created version '3' of model 'mlops_dev.nikhilko.model_fe_demo': https://dbc-f122dc18-1b68.cloud.databricks.com/explore/data/models/mlops_dev/nikhilko/model_fe_demo/version/3?o=2661948581729539


In [0]:
# make predictions
features = [f for f in ["Booking_ID"] + config.num_features + config.cat_features if f not in lookup_features]
predictions = fe.score_batch(model_uri=f"models:/{model_name}/{model_version.version}", df=test_set[features])

2025/09/24 07:26:26 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/09/24 07:26:26 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [0]:
predictions.select("prediction").show(5)

+----------+
|prediction|
+----------+
|       0.0|
|       0.0|
|       1.0|
|       0.0|
|       1.0|
+----------+
only showing top 5 rows


In [0]:
features = [f for f in ["Booking_ID"] + config.num_features + config.cat_features if f not in lookup_features]
test_set_with_new_id = test_set.select(*features)
# .withColumn(
#     "Booking_ID",
#     (col("Booking_ID").cast("long") + 1000000).cast("string")
# )

predictions = fe.score_batch(model_uri=f"models:/{model_name}/{model_version.version}", df=test_set_with_new_id)

2025/09/24 07:27:14 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/09/24 07:27:14 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [0]:
# make predictions for a non-existing entry -> error!
predictions.select("prediction").show(5)

+----------+
|prediction|
+----------+
|       0.0|
|       0.0|
|       1.0|
|       0.0|
|       1.0|
+----------+
only showing top 5 rows


In [0]:
no_of_previous_cancellations_function = (
    f"{config.catalog_name}.{config.schema_name}.replace_no_of_previous_cancellations_missing"
)
spark.sql(f"""
        CREATE OR REPLACE FUNCTION {no_of_previous_cancellations_function}(no_of_previous_cancellations BIGINT)
        RETURNS BIGINT
        LANGUAGE PYTHON AS
        $$
        if no_of_previous_cancellations is None:
            return 0
        else:
            return no_of_previous_cancellations
        $$
        """)

no_of_previous_bookings_not_canceled_function = (
    f"{config.catalog_name}.{config.schema_name}.replace_no_of_previous_bookings_not_canceled_missing"
)
spark.sql(f"""
        CREATE OR REPLACE FUNCTION {no_of_previous_bookings_not_canceled_function}(no_of_previous_bookings_not_canceled BIGINT)
        RETURNS BIGINT
        LANGUAGE PYTHON AS
        $$
        if no_of_previous_bookings_not_canceled is None:
            return 1
        else:
            return no_of_previous_bookings_not_canceled
        $$
        """)

DataFrame[]

In [0]:
# what if we want to replace with a default value if entry is not found
# what if we want to look up value in another table? the logics get complex
# problems that arize: functions/ lookups always get executed (if statememt is not possible)
# it can get slow...

# step 1: create 3 feature functions

# step 2: redefine create training set

# try again

# create a training set
training_set = fe.create_training_set(
    df=train_set.drop("no_of_previous_cancellations", "no_of_previous_bookings_not_canceled"),
    label=config.target,
    feature_lookups=[
        FeatureLookup(
            table_name=feature_table_name,
            feature_names=["no_of_previous_cancellations", "no_of_previous_bookings_not_canceled"],
            lookup_key="Booking_ID",
            rename_outputs={
                "no_of_previous_cancellations": "lookup_no_of_previous_cancellations",
                "no_of_previous_bookings_not_canceled": "lookup_no_of_previous_bookings_not_canceled",
            },
        ),
        FeatureFunction(
            udf_name=no_of_previous_cancellations_function,
            output_name="no_of_previous_cancellations",
            input_bindings={"no_of_previous_cancellations": "lookup_no_of_previous_cancellations"},
        ),
        FeatureFunction(
            udf_name=no_of_previous_bookings_not_canceled_function,
            output_name="no_of_previous_bookings_not_canceled",
            input_bindings={"no_of_previous_bookings_not_canceled": "lookup_no_of_previous_bookings_not_canceled"},
        ),
        FeatureFunction(
            udf_name=function_name,
            output_name="booking_value",
            input_bindings={
                "avg_price_per_room": "avg_price_per_room",
                "no_of_weekend_nights": "no_of_weekend_nights",
                "no_of_week_nights": "no_of_week_nights",
            },
        ),
    ],
    exclude_columns=["update_timestamp_utc"],
)

In [0]:
# Train & register a model
training_df = training_set.load_df().toPandas()
X_train = training_df[config.num_features + config.cat_features + ["booking_value"]]
y_train = training_df[config.target]
y_train_encoded = labelEncoder.fit_transform(y_train)

# pipeline
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), config.cat_features)],
                remainder="passthrough",
            ),
        ),
        ("classifier", LGBMClassifier(**config.parameters)),
    ]
)

pipeline.fit(X_train, y_train_encoded)

[LightGBM] [Info] Number of positive: 19551, number of negative: 9469
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003330 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 915
[LightGBM] [Info] Number of data points in the train set: 29020, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.673708 -> initscore=0.725003
[LightGBM] [Info] Start training from score 0.725003
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [0]:
mlflow.set_experiment("/Shared/demo-model-fe")
with mlflow.start_run(
    run_name="demo-run-model-fe",
    tags={"git_sha": "1234567890abcd", "branch": "week2"},
    description="demo run for FE model logging",
) as run:
    # Log parameters and metrics
    run_id = run.info.run_id
    mlflow.log_param("model_type", "LightGBM with preprocessing")
    mlflow.log_params(config.parameters)

    # Log the model
    signature = infer_signature(model_input=X_train, model_output=pipeline.predict(X_train))
    fe.log_model(
        model=pipeline,
        flavor=mlflow.sklearn,
        artifact_path="lightgbm-pipeline-model-fe",
        training_set=training_set,
        signature=signature,
    )
model_name = f"{config.catalog_name}.{config.schema_name}.model_fe_demo"
model_version = mlflow.register_model(
    model_uri=f"runs:/{run_id}/lightgbm-pipeline-model-fe", name=model_name, tags={"git_sha": "1234567890abcd"}
)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.h

Uploading artifacts:   0%|          | 0/15 [00:00<?, ?it/s]

🔗 Created version '4' of model 'mlops_dev.nikhilko.model_fe_demo': https://dbc-f122dc18-1b68.cloud.databricks.com/explore/data/models/mlops_dev/nikhilko/model_fe_demo/version/4?o=2661948581729539


In [0]:
features = [f for f in ["Booking_ID"] + config.num_features + config.cat_features if f not in lookup_features]
test_set_with_new_id = test_set.select(*features)
# .withColumn(
#     "Booking_ID",
#     (col("Id").cast("long") + 1000000).cast("string")
# )

predictions = fe.score_batch(model_uri=f"models:/{model_name}/{model_version.version}", df=test_set_with_new_id)

2025/09/24 07:33:36 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/09/24 07:33:36 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [0]:
# make predictions for a non-existing entry -> no error!
predictions.select("prediction").show(5)

+----------+
|prediction|
+----------+
|       0.0|
|       0.0|
|       1.0|
|       0.0|
|       1.0|
+----------+
only showing top 5 rows


In [0]:
dbutils.secrets.get(scope="mlops", key="aws_access_key_id")

'[REDACTED]'

In [0]:
dbutils.secrets.get(scope="mlops", key="aws_secret_access_key")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-4551535645479341>, line 1
----> 1 dbutils.secrets.get(scope="mlops", key="aws_secret_access_key")

File /databricks/python_shell/lib/dbruntime/dbutils.py:359, in DBUtils.SecretsHandler.get(self, scope, key)
    357 def get(self, scope, key):
    358     return self.entry_point.getDbutils().preview().secret(
--> 359     ).get(  # type: ignore[attr-defined]
    360         scope, key)

File /databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py:1362, in JavaMember.__call__(self, *args)
   1356 command = proto.CALL_COMMAND_NAME +\
   1357     self.command_header +\
   1358     args_command +\
   1359     proto.END_COMMAND_PART
   1361 answer = self.gateway_client.send_command(command)
-> 1362 return_value = get_return_value(
   1363     answer, self.gateway_client, self.target_id, self.name)
   1365 for temp_

In [0]:
dbutils.secrets.list("mlops")

[SecretMetadata(key='aws_access_key'),
 SecretMetadata(key='aws_access_key_id'),
 SecretMetadata(key='GITHUB_TOKEN')]

In [0]:
region_name = "eu-west-1"
aws_access_key_id = dbutils.secrets.get(scope="mlops", key="aws_access_key_id")  # os.environ["aws_access_key_id"]
aws_secret_access_key = dbutils.secrets.get(scope="mlops", key="aws_access_key")  # os.environ["aws_secret_access_key"]

client = boto3.client(
    "dynamodb",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name,
)

In [0]:
response = client.create_table(
    TableName="HotelFeatures",
    KeySchema=[
        {
            "AttributeName": "Booking_ID",
            "KeyType": "HASH",  # Partition key
        }
    ],
    AttributeDefinitions=[
        {
            "AttributeName": "Booking_ID",
            "AttributeType": "S",  # String
        }
    ],
    ProvisionedThroughput={"ReadCapacityUnits": 5, "WriteCapacityUnits": 5},
)

print("Table creation initiated:", response["TableDescription"]["TableName"])

Table creation initiated: HotelFeatures


In [0]:
client.put_item(
    TableName="HotelFeatures",
    Item={
        "Booking_ID": {"S": "hotel_001"},
        "no_of_previous_cancellations": {"N": "8"},
        "no_of_previous_bookings_not_canceled": {"N": "2450"},
    },
)

{'ResponseMetadata': {'RequestId': 'E5S2K0R3UTM6UFKU9UOLCSUNL3VV4KQNSO5AEMVJF66Q9ASUAAJG',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Wed, 24 Sep 2025 07:47:33 GMT',
   'content-type': 'application/x-amz-json-1.0',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'E5S2K0R3UTM6UFKU9UOLCSUNL3VV4KQNSO5AEMVJF66Q9ASUAAJG',
   'x-amz-crc32': '2745614147'},
  'RetryAttempts': 0}}

In [0]:
response = client.get_item(TableName="HotelFeatures", Key={"Booking_ID": {"S": "hotel_001"}})

# Extract the item from the response
item = response.get("Item")
print(item)

{'Booking_ID': {'S': 'hotel_001'}, 'no_of_previous_cancellations': {'N': '8'}, 'no_of_previous_bookings_not_canceled': {'N': '2450'}}


In [0]:
rows = spark.table(feature_table_name).toPandas().to_dict(orient="records")


def to_dynamodb_item(row):
    return {
        "PutRequest": {
            "Item": {
                "Booking_ID": {"S": str(row["Booking_ID"])},
                "no_of_previous_cancellations": {"N": str(row["no_of_previous_cancellations"])},
                "no_of_previous_bookings_not_canceled": {"N": str(row["no_of_previous_bookings_not_canceled"])},
            }
        }
    }


items = [to_dynamodb_item(row) for row in rows]


def chunks(lst, n):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


for batch in chunks(items, 25):
    response = client.batch_write_item(RequestItems={"HotelFeatures": batch})
    # Handle any unprocessed items if needed
    unprocessed = response.get("UnprocessedItems", {})
    if unprocessed:
        print("Warning: Some items were not processed. Retry logic needed.")

In [0]:
# We ran into more limitations when we tried complex data types as output of a feature function
# and then tried to use it for serving
# al alternatve solution: using an external database (we use DynamoDB here)

# create a DynamoDB table
# insert records into dynamo DB & read from dynamoDB

# create a pyfunc model

In [0]:
class HotelReservationModelWrapper(mlflow.pyfunc.PythonModel):
    """Wrapper class for machine learning models to be used with MLflow.

    This class wraps a machine learning model for predicting hotel reservation's booking status.
    """

    def __init__(self, model: object) -> None:
        """Initialize the HotelReservationModelWrapper.

        :param model: The underlying machine learning model.
        """
        self.model = model

    def predict(
        self, context: mlflow.pyfunc.PythonModelContext, model_input: pd.DataFrame | np.ndarray
    ) -> dict[str, float]:
        """Make predictions using the wrapped model.

        :param context: The MLflow context (unused in this implementation).
        :param model_input: Input data for making predictions.
        :return: A dictionary containing the adjusted prediction.
        """
        client = boto3.client(
            "dynamodb",
            aws_access_key_id=dbutils.secrets.get(scope="mlops", key="aws_access_key_id"),
            aws_secret_access_key=dbutils.secrets.get(scope="mlops", key="aws_access_key"),
            region_name=region_name,
        )

        parsed = []
        for lookup_id in model_input["Booking_ID"]:
            raw_item = client.get_item(TableName="HotelFeatures", Key={"Booking_ID": {"S": lookup_id}})["Item"]
            parsed_dict = {key: int(value["N"]) if "N" in value else value["S"] for key, value in raw_item.items()}
            parsed.append(parsed_dict)
        lookup_df = pd.DataFrame(parsed)
        merged_df = model_input.merge(lookup_df, on="Booking_ID", how="left").drop("Booking_ID", axis=1)

        merged_df["no_of_previous_cancellations"] = merged_df["no_of_previous_cancellations"].fillna(2)
        merged_df["no_of_previous_bookings_not_canceled"] = merged_df["no_of_previous_bookings_not_canceled"].fillna(2)
        merged_df["booking_value"] = merged_df["avg_price_per_room"] * (
            merged_df["no_of_weekend_nights"] + merged_df["no_of_week_nights"]
        )
        predictions = self.model.predict(merged_df)

        return [int(x) for x in predictions]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/mlflow/pyfunc/model.py:185: UserWarning: Type hint used in the model's predict function is not supported for MLflow's schema validation. Type hints must be wrapped in list[...] because MLflow assumes the predict method to take multiple input instances. Specify your type hint as `list[pandas.core.frame.DataFrame | numpy.ndarray]` for a valid signature. Remove the type hint to disable this warning. To enable validation for the input data, specify input example or model signature when logging the model. 
  func_info = _get_func_info_if_type_hint_supported(predict_attr)


In [0]:
custom_model = HotelReservationModelWrapper(pipeline)

In [0]:
features = [f for f in ["Booking_ID"] + config.num_features + config.cat_features if f not in lookup_features]
data = test_set.select(*features).toPandas()
data

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,arrival_year,arrival_month,arrival_date,repeated_guest,avg_price_per_room,no_of_special_requests,type_of_meal_plan,room_type_reserved,market_segment_type
0,INN17698,1,0,1,0,0,27,2017,11,29,0,65.00,0,Meal Plan 1,Room_Type 1,Corporate
1,INN16234,2,0,0,4,0,40,2018,12,7,0,96.90,1,Meal Plan 1,Room_Type 4,Online
2,INN28112,2,0,0,2,0,48,2018,2,11,0,89.30,0,Meal Plan 1,Room_Type 1,Online
3,INN15336,2,0,1,2,0,305,2018,11,4,0,89.00,0,Meal Plan 1,Room_Type 1,Offline
4,INN23760,2,0,2,2,0,28,2018,5,1,0,133.95,1,Meal Plan 1,Room_Type 1,Online
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7250,INN02768,2,0,0,2,0,86,2018,5,26,0,140.40,0,Meal Plan 1,Room_Type 4,Online
7251,INN29523,1,0,0,2,0,39,2017,8,14,0,87.00,0,Meal Plan 2,Room_Type 1,Offline
7252,INN25061,2,0,1,0,0,20,2018,6,26,0,89.00,1,Not Selected,Room_Type 1,Online
7253,INN08475,2,0,0,3,0,308,2018,11,23,0,78.30,2,Meal Plan 1,Room_Type 1,Online


In [0]:
custom_model.predict(context=None, model_input=data)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[1,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,


In [0]:
# log model
mlflow.set_experiment("/Shared/demo-model-fe-pyfunc")
with mlflow.start_run(
    run_name="demo-run-model-fe-pyfunc",
    tags={"git_sha": "1234567890abcd", "branch": "week2"},
    description="demo run for FE model logging",
) as run:
    # Log parameters and metrics
    run_id = run.info.run_id
    mlflow.log_param("model_type", "LightGBM with preprocessing")
    mlflow.log_params(config.parameters)

    # Log the model
    signature = infer_signature(model_input=data, model_output=custom_model.predict(context=None, model_input=data))
    mlflow.pyfunc.log_model(
        python_model=custom_model,
        artifact_path="lightgbm-pipeline-model-fe-custom",
        signature=signature,
    )

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.h

 You cannot use dbutils within a spark job or otherwise pickle it.
            If you need to use getArguments within a spark job, you have to get the argument before
            using it in the job. For example, if you have the following code:

              myRdd.map(lambda i: dbutils.args.getArgument("X") + str(i))

            Then you should use it this way:

              argX = dbutils.args.getArgument("X")
              myRdd.map(lambda i: argX + str(i))
            


---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File /local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/mlflow/pyfunc/model.py:1038, in _save_model_with_class_artifacts_params(path, python_model, signature, artifacts, conda_env, code_paths, mlflow_model, pip_requirements, extra_pip_requirements, model_config, streamable, model_code_path, infer_code_paths)
   1037 try:
-> 1038     _maybe_compress_cloudpickle_dump(
   1039         python_model, os.path.join(path, saved_python_model_subpath), compression
   1040     )
   1041 except Exception as e:

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-c37a5dfb-fa5e-43ec-8d36-0c8b526fd57e/lib/python3.12/site-packages/mlflow/pyfunc/model.py:817, in _maybe_compress_cloudpickle_dump(python_model, path, compression)
    816 with file_open(path, "wb") as out:
--> 817     cloudpickle.dump(py

In [0]:
# predict
mlflow.models.predict(f"runs:/{run_id}/lightgbm-pipeline-model-fe", data[0:1])

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can